# LeetCode 140 — Word Break II (Hard) bằng Python

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def wordBreak(self, s: str, wordDict: List[str]) -> List[str]:

```

**Mục tiêu sau khi học xong:**

1. Hiểu đề: "chèn dấu cách" vào chuỗi để mọi mảnh đều là từ trong từ điển, và phải trả về **tất cả** các câu.
2. Ôn lại **Word Break I (LeetCode 139)** — phiên bản chỉ hỏi `True/False` — bằng **DP (quy hoạch động)** từ căn bản.
3. Học **đệ quy / backtracking (quay lui)** từ con số 0 và viết **Cách 1 — thử mọi tiền tố thuộc từ điển**.
4. Thấy Cách 1 "chết" ở đâu (nhánh chết bị đi lại nhiều lần) và sửa bằng **Cách 2 — backtracking + cắt tỉa bằng `can_break`**.
5. Học **memoization** và viết **Cách 3 — đệ quy + memo trả về list câu cho mỗi vị trí** — đây là **cách chính** để nộp.
6. Biết **Cách 4 — DP bottom-up `dp[i]` = list câu của `s[:i]`** và hiểu vì sao nó có thể làm phí công.
7. Biết **Cách 5 — dùng Trie (cây tiền tố)** để duyệt tiền tố nhanh hơn khi từ điển lớn (tùy chọn).
8. Hiểu vì sao **đầu ra có thể lớn theo hàm mũ**, nên **không có thuật toán đa thức tổng quát** — chỉ có thuật toán "nhạy với đầu ra" (output-sensitive).

## 1. Đề bài

Cho một chuỗi `s` và một từ điển là danh sách chuỗi `wordDict`. Hãy **thêm dấu cách** vào `s` để tạo thành một câu mà **mỗi từ đều là một từ hợp lệ trong từ điển**. Trả về **tất cả** các câu có thể, theo **thứ tự bất kỳ**.

**Lưu ý:** cùng một từ trong từ điển có thể được **dùng lại nhiều lần** khi tách.

**Ví dụ 1**

```text
Input:  s = "catsanddog", wordDict = ["cat","cats","and","sand","dog"]
Output: ["cats and dog","cat sand dog"]
```

**Ví dụ 2**

```text
Input:  s = "pineapplepenapple", wordDict = ["apple","pen","applepen","pine","pineapple"]
Output: ["pine apple pen apple","pineapple pen apple","pine applepen apple"]
Explanation: Note that you are allowed to reuse a dictionary word.
```

**Ví dụ 3**

```text
Input:  s = "catsandog", wordDict = ["cats","dog","sand","and","cat"]
Output: []
```

**Constraints (ràng buộc):**

- `1 <= s.length <= 20`
- `1 <= wordDict.length <= 1000`
- `1 <= wordDict[i].length <= 10`
- `s` và `wordDict[i]` chỉ gồm chữ cái tiếng Anh viết thường.
- Mọi chuỗi trong `wordDict` là **khác nhau** (unique).
- Input được sinh sao cho **độ dài của đáp án không vượt quá `10^5`**.

Dòng ràng buộc cuối rất quan trọng — ta sẽ hiểu vì sao đề phải "hứa" như vậy ở mục 9.

### Đề nói "thứ tự bất kỳ" — vậy test thế nào?

LeetCode chấp nhận các câu theo **mọi thứ tự**. Vì thế khi tự test, ta **không** so sánh hai list trực tiếp (vì `["a b", "c"] != ["c", "a b"]`), mà so sánh sau khi **sắp xếp** cả hai bằng `sorted(...)`. Cell dưới lưu 3 ví dụ của đề và viết sẵn hàm nhỏ `same_answer` để dùng trong cả notebook.

In [1]:
from typing import List

examples = [
    ("catsanddog", ["cat", "cats", "and", "sand", "dog"],
     ["cats and dog", "cat sand dog"]),
    ("pineapplepenapple", ["apple", "pen", "applepen", "pine", "pineapple"],
     ["pine apple pen apple", "pineapple pen apple", "pine applepen apple"]),
    ("catsandog", ["cats", "dog", "sand", "and", "cat"],
     []),
]

for s, words, expected in examples:
    print(f"s = {s!r}")
    print(f"   wordDict = {words}")
    print(f"   expected = {expected}")


def same_answer(actual: List[str], expected: List[str]) -> bool:
    # So sánh 2 list câu, bỏ qua thứ tự
    return sorted(actual) == sorted(expected)

print(["cats and dog", "cat sand dog"] == ["cat sand dog", "cats and dog"])            # False
print(same_answer(["cats and dog", "cat sand dog"], ["cat sand dog", "cats and dog"]))  # True

s = 'catsanddog'
   wordDict = ['cat', 'cats', 'and', 'sand', 'dog']
   expected = ['cats and dog', 'cat sand dog']
s = 'pineapplepenapple'
   wordDict = ['apple', 'pen', 'applepen', 'pine', 'pineapple']
   expected = ['pine apple pen apple', 'pineapple pen apple', 'pine applepen apple']
s = 'catsandog'
   wordDict = ['cats', 'dog', 'sand', 'and', 'cat']
   expected = []
False
True


## 2. Kiến thức nền Python cho bài này

### 2.1. Cắt chuỗi (slicing): tiền tố và hậu tố

- `s[a:b]` lấy các ký tự từ chỉ số `a` **đến trước** `b` (không lấy `b`).
- **Tiền tố (prefix)** của `s` là một đoạn đầu: `s[:i]`.
- **Hậu tố (suffix)** là một đoạn cuối: `s[i:]`.

Với `s = "catsanddog"` (chỉ số `0..9`):

| i | `s[:i]` (tiền tố) | `s[i:]` (hậu tố) |
|---:|---|---|
| 3 | `"cat"` | `"sanddog"` |
| 4 | `"cats"` | `"anddog"` |
| 7 | `"catsand"` | `"dog"` |
| 10 | `"catsanddog"` | `""` (chuỗi rỗng) |

Ý tưởng cốt lõi của mọi lời giải: **chọn từ đầu tiên** là một tiền tố thuộc từ điển, rồi **giải tiếp phần hậu tố còn lại**.

In [2]:
s = "catsanddog"
for i in [3, 4, 7, 10]:
    print(f"i = {i:>2} | tiền tố s[:{i}] = {s[:i]!r:<14} | hậu tố s[{i}:] = {s[i:]!r}")

# Một đoạn ở giữa: từ vị trí start, dài tới end (không lấy end)
start, end = 3, 7
print("s[3:7] =", s[start:end])

i =  3 | tiền tố s[:3] = 'cat'          | hậu tố s[3:] = 'sanddog'
i =  4 | tiền tố s[:4] = 'cats'         | hậu tố s[4:] = 'anddog'
i =  7 | tiền tố s[:7] = 'catsand'      | hậu tố s[7:] = 'dog'
i = 10 | tiền tố s[:10] = 'catsanddog'   | hậu tố s[10:] = ''
s[3:7] = sand


### 2.2. Tra từ điển bằng `set` thay vì `list`

Ta sẽ hỏi "đoạn này có phải là từ không?" **rất nhiều lần**. Với `list`, câu hỏi `w in list` phải dò từng phần tử: tốn `O(số từ)` lần so sánh. Với `set` (tập hợp, dùng bảng băm — hash table), câu hỏi `w in set` chỉ tốn trung bình `O(độ dài w)` để băm chuỗi, **không phụ thuộc** số từ trong từ điển.

Từ điển có thể tới 1000 từ, nên đổi `wordDict` sang `set` ngay từ đầu là thói quen tốt.

### 2.3. Ghép list từ thành câu bằng `" ".join(...)`

Đề trả về **câu** (chuỗi có dấu cách), không phải list từ. `" ".join(words)` nối các phần tử bằng **đúng một** dấu cách ở **giữa** — không thừa ở đầu hay cuối.

Một điểm tinh tế: `" ".join([])` cho ra chuỗi rỗng `""`. Ta sẽ gặp lại chuỗi rỗng này ở base case của đệ quy.

In [3]:
wordDict = ["cat", "cats", "and", "sand", "dog"]
word_set = set(wordDict)

print(sorted(word_set))     # set không có thứ tự cố định -> in bản đã sắp xếp cho dễ nhìn
print("cat" in word_set)    # True
print("ca" in word_set)     # False  -> "ca" chỉ là một phần của từ, không phải từ
print("sand" in word_set)   # True

# Liệt kê mọi tiền tố của "catsanddog" là từ trong từ điển
s = "catsanddog"
for end in range(1, len(s) + 1):
    prefix = s[:end]
    if prefix in word_set:
        print("tiền tố là từ:", prefix)

# 2.3. Ghép từ thành câu
print(repr(" ".join(["cat", "sand", "dog"])))  # 'cat sand dog'
print(repr(" ".join(["dog"])))                 # 'dog'  (1 từ: không có dấu cách)
print(repr(" ".join([])))                      # ''     (0 từ: chuỗi rỗng)

# Ghép "từ đầu" + "phần câu còn lại"
first, rest = "cat", "sand dog"
print(repr(first + " " + rest))

['and', 'cat', 'cats', 'dog', 'sand']
True
False
True
tiền tố là từ: cat
tiền tố là từ: cats
'cat sand dog'
'dog'
''
'cat sand dog'


## 3. Ôn lại Word Break I (LeetCode 139) — chỉ hỏi "cắt được không?"

**Word Break I**: cùng input `s`, `wordDict`, nhưng chỉ trả về `True` nếu `s` **có thể** tách thành các từ trong từ điển, ngược lại `False`.

Vì sao học bài dễ này trước?

- Nó có **cùng cấu trúc** với bài II: "chọn từ đầu tiên, giải phần còn lại".
- Kết quả `True/False` của nó sẽ là **công cụ cắt tỉa** cho Cách 2 của bài II.
- Nó là ví dụ DP đẹp để làm quen trước khi DP "lưu cả list câu".

### DP là gì? (nói đơn giản)

**Quy hoạch động (Dynamic Programming — DP)** = chia bài lớn thành các **bài con** nhỏ hơn, giải mỗi bài con **một lần**, lưu kết quả vào bảng, rồi dùng lại.

Với Word Break I, định nghĩa:

> `dp[i] = True` nếu **tiền tố** `s[:i]` (i ký tự đầu) tách được thành các từ.

- `dp[0] = True`: tiền tố rỗng "tách được" (dùng 0 từ). Đây là **điểm tựa** để bắt đầu.
- Với `i ≥ 1`: `dp[i] = True` nếu tồn tại `j < i` sao cho `dp[j] = True` **và** `s[j:i]` là một từ.

Nói bằng lời: "`s[:i]` tách được nếu ta tách được một đoạn đầu `s[:j]`, và đoạn cuối `s[j:i]` là một từ trọn vẹn."

**Tính tay với `s = "catsanddog"`:**

| i | `s[:i]` | Có `j` nào thỏa? | `dp[i]` |
|---:|---|---|:---:|
| 0 | `""` | (điểm tựa) | T |
| 3 | `"cat"` | `j=0`: `dp[0]=T`, `"cat"` là từ | T |
| 4 | `"cats"` | `j=0`: `"cats"` là từ | T |
| 7 | `"catsand"` | `j=3`: `dp[3]=T`, `"sand"` là từ (cũng có `j=4`: `"and"`) | T |
| 10 | `"catsanddog"` | `j=7`: `dp[7]=T`, `"dog"` là từ | T |

Các `i` còn lại (1, 2, 5, 6, 8, 9) đều `False`.

In [4]:
def word_break_i_dp(s: str, wordDict: List[str]) -> bool:
    word_set = set(wordDict)
    n = len(s)
    dp = [False] * (n + 1)
    dp[0] = True                      # tiền tố rỗng: tách được bằng 0 từ
    for i in range(1, n + 1):
        for j in range(i):
            if dp[j] and s[j:i] in word_set:
                dp[i] = True
                break                 # chỉ cần 1 cách là đủ
    return dp[n]

print(word_break_i_dp("catsanddog", ["cat", "cats", "and", "sand", "dog"]))  # True
print(word_break_i_dp("catsandog", ["cats", "dog", "sand", "and", "cat"]))   # False
print(word_break_i_dp("applepenapple", ["apple", "pen"]))                    # True

True
False
True


### Trace Word Break I: vì sao `"catsandog"` lại `False`?

Hàm `trace_word_break_i` in ra, với mỗi `i` có `dp[i] = True`, cái `j` và từ `s[j:i]` đã "mở khóa" nó. Với `"catsandog"`, ta đi được tới `dp[7]` (`"catsand"`), nhưng phần cuối `"og"`, `"g"`, `"dog"`... không có đoạn nào vừa là từ vừa bắt đầu ở một vị trí `True` — nên `dp[9]` vẫn `False`.

In [5]:
def trace_word_break_i(s: str, wordDict: List[str]) -> bool:
    word_set = set(wordDict)
    n = len(s)
    dp = [False] * (n + 1)
    dp[0] = True
    for i in range(1, n + 1):
        for j in range(i):
            if dp[j] and s[j:i] in word_set:
                dp[i] = True
                print(f"dp[{i:>2}] = True  vì dp[{j}] = True và s[{j}:{i}] = {s[j:i]!r} là từ")
                break
    print("Bảng dp:", "".join("T" if x else "." for x in dp), "-> kết quả", dp[n])
    return dp[n]

trace_word_break_i("catsanddog", ["cat", "cats", "and", "sand", "dog"])
print()
trace_word_break_i("catsandog", ["cats", "dog", "sand", "and", "cat"])

dp[ 3] = True  vì dp[0] = True và s[0:3] = 'cat' là từ
dp[ 4] = True  vì dp[0] = True và s[0:4] = 'cats' là từ
dp[ 7] = True  vì dp[3] = True và s[3:7] = 'sand' là từ
dp[10] = True  vì dp[7] = True và s[7:10] = 'dog' là từ
Bảng dp: T..TT..T..T -> kết quả True

dp[ 3] = True  vì dp[0] = True và s[0:3] = 'cat' là từ
dp[ 4] = True  vì dp[0] = True và s[0:4] = 'cats' là từ
dp[ 7] = True  vì dp[3] = True và s[3:7] = 'sand' là từ
Bảng dp: T..TT..T.. -> kết quả False


False

### Word Break I "nhìn từ phía sau": `can_break[i]` cho **hậu tố**

Cách 2 của bài II cần trả lời câu hỏi: "**từ vị trí `i` trở đi**, phần còn lại `s[i:]` có tách được không?". Đây là Word Break I nhưng tính cho **hậu tố**, đi từ phải sang trái:

- `can_break[n] = True` (hậu tố rỗng).
- `can_break[i] = True` nếu có `end > i` mà `s[i:end]` là từ **và** `can_break[end] = True`.

Ví dụ `"catsandog"`: `can_break[7]` (hậu tố `"og"`) là `False`, `can_break[3]` (`"sandog"`) cũng `False` vì sau `"sand"` chỉ còn `"og"`... và cuối cùng `can_break[0] = False`.

In [6]:
def build_can_break(s: str, word_set: set) -> List[bool]:
    n = len(s)
    can = [False] * (n + 1)
    can[n] = True                              # hậu tố rỗng
    for i in range(n - 1, -1, -1):             # đi từ phải sang trái
        for end in range(i + 1, n + 1):
            if can[end] and s[i:end] in word_set:
                can[i] = True
                break
    return can

for s, words in [("catsanddog", ["cat", "cats", "and", "sand", "dog"]),
                 ("catsandog", ["cats", "dog", "sand", "and", "cat"])]:
    can = build_can_break(s, set(words))
    print(f"{s:<11}", "chỉ số:", " ".join(f"{i:>2}" for i in range(len(s) + 1)))
    print(f"{'':<11}", "can   :", " ".join(f"{'T' if c else '.':>2}" for c in can))

catsanddog  chỉ số:  0  1  2  3  4  5  6  7  8  9 10
            can   :  T  .  .  T  T  .  .  T  .  .  T
catsandog   chỉ số:  0  1  2  3  4  5  6  7  8  9
            can   :  .  .  .  .  .  .  T  .  .  T


## 4. Kiến thức nền: đệ quy và backtracking (quay lui)

### 4.1. Đệ quy (recursion) là gì?

**Đệ quy** = một hàm **tự gọi lại chính nó** với một bài **nhỏ hơn**, cho tới khi gặp **trường hợp cơ sở (base case)** đủ nhỏ để trả lời ngay.

### 4.2. Backtracking là gì?

**Backtracking (quay lui)** là cách duyệt mọi lựa chọn theo kiểu "đi thử một đường, đi hết thì **lùi lại** thử đường khác":

1. **Chọn** một lựa chọn → ghi vào `path` (đường đi hiện tại) bằng `path.append(...)`.
2. **Đi tiếp** bằng lời gọi đệ quy.
3. **Hoàn tác** lựa chọn: `path.pop()`, để thử lựa chọn kế tiếp với `path` sạch.

### Ví dụ mini: liệt kê **mọi cách** cắt chuỗi `"abcd"` thành các mảnh

Chưa cần từ điển: mảnh nào cũng được. Tại vị trí `start`, ta chọn mảnh đầu `s[start:end]` với mọi `end`, rồi cắt tiếp phần còn lại. Base case: `start == len(s)` (đã cắt hết) → ghi lại một cách cắt.

In [7]:
def all_splits(s: str) -> List[List[str]]:
    result = []
    path = []

    def go(start: int) -> None:
        if start == len(s):              # base case: đã cắt hết chuỗi
            result.append(path.copy())   # copy! vì path còn bị sửa tiếp
            return
        for end in range(start + 1, len(s) + 1):
            path.append(s[start:end])    # 1. chọn mảnh đầu
            go(end)                      # 2. cắt tiếp phần còn lại
            path.pop()                   # 3. hoàn tác để thử mảnh khác

    go(0)
    return result

splits = all_splits("abcd")
for parts in splits:
    print(parts)
print("Số cách cắt:", len(splits))

['a', 'b', 'c', 'd']
['a', 'b', 'cd']
['a', 'bc', 'd']
['a', 'bcd']
['ab', 'c', 'd']
['ab', 'cd']
['abc', 'd']
['abcd']
Số cách cắt: 8


### Có bao nhiêu cách cắt một chuỗi dài `n`?

Giữa `n` ký tự có `n - 1` "khe hở". Mỗi khe ta **cắt** hoặc **không cắt** → có `2^(n-1)` cách. Với `"abcd"` (`n = 4`): `2^3 = 8` cách — đúng như output trên.

Với `n = 20` (giới hạn của đề): `2^19 = 524 288` cách. Con số này **tăng gấp đôi** mỗi khi thêm 1 ký tự — đó là **tăng theo hàm mũ (exponential)**.

## 5. Cách 0 — Brute force thuần: sinh mọi cách cắt rồi lọc

Cách mà người mới nghĩ ra đầu tiên: dùng `all_splits`, sau đó giữ lại những cách cắt mà **mọi mảnh** đều là từ.

- **Ưu điểm**: chắc chắn đúng, rất dễ hiểu → dùng làm **"trọng tài"** để kiểm tra các cách khác.
- **Nhược điểm**: luôn sinh đủ `2^(n-1)` cách cắt, kể cả khi mảnh đầu tiên đã sai từ sớm (ví dụ `"c"` không phải từ mà vẫn cắt tiếp phần sau).

In [8]:
def word_break_brute(s: str, wordDict: List[str]) -> List[str]:
    word_set = set(wordDict)
    sentences = []
    for parts in all_splits(s):
        if all(p in word_set for p in parts):
            sentences.append(" ".join(parts))
    return sentences

for s, words, expected in examples:
    got = word_break_brute(s, words)
    print(f"{s:<18} -> {got}")
    assert same_answer(got, expected)

print("Số cách cắt phải sinh ra cho 'catsanddog':", len(all_splits("catsanddog")))

catsanddog         -> ['cat sand dog', 'cats and dog']
pineapplepenapple  -> ['pine apple pen apple', 'pine applepen apple', 'pineapple pen apple']
catsandog          -> []
Số cách cắt phải sinh ra cho 'catsanddog': 512


## 6. Cách 1 — Backtracking: chỉ đi tiếp khi tiền tố là **từ**

### Cách 0 vướng ở đâu?

Cách 0 cắt ra `"c"`, rồi vẫn hăng hái cắt tiếp `"atsanddog"` theo đủ mọi kiểu, dù **ngay từ đầu** `"c"` đã không phải từ. Mọi cách cắt bắt đầu bằng `"c"` đều vô dụng.

### Mấu chốt cải tiến

Kiểm tra **ngay khi chọn** mảnh: chỉ khi `s[start:end]` **là từ** thì mới đi tiếp vào `go(end)`. Đây chính là "backtracking có điều kiện" — ta cắt bỏ cả một **cây con** lựa chọn vô dụng chỉ bằng một phép `in`.

```python
for end in range(start + 1, n + 1):
    word = s[start:end]
    if word in word_set:      # <- chỉ khác Cách 0 ở dòng này
        path.append(word)
        go(end)
        path.pop()
```

Khi `start == n`, `path` chứa đúng một câu hợp lệ → `" ".join(path)` và lưu lại.

In [9]:
def word_break_backtrack(s: str, wordDict: List[str]) -> List[str]:
    word_set = set(wordDict)
    n = len(s)
    result = []
    path = []

    def go(start: int) -> None:
        if start == n:
            result.append(" ".join(path))
            return
        for end in range(start + 1, n + 1):
            word = s[start:end]
            if word in word_set:
                path.append(word)
                go(end)
                path.pop()

    go(0)
    return result

for s, words, expected in examples:
    got = word_break_backtrack(s, words)
    print(f"{s:<18} -> {got}")
    assert same_answer(got, expected)

catsanddog         -> ['cat sand dog', 'cats and dog']
pineapplepenapple  -> ['pine apple pen apple', 'pine applepen apple', 'pineapple pen apple']
catsandog          -> []


### Trace Cách 1: cây quyết định của `"catsanddog"` và `"catsandog"`

Hàm `trace_backtrack` in mỗi lần **chọn một từ** (thụt lề theo độ sâu đệ quy), mỗi lần **ra được câu** (✔), và mỗi khi đứng ở một vị trí mà **không có từ nào** bắt đầu từ đó (✘ nhánh chết).

In [10]:
def trace_backtrack(s: str, wordDict: List[str]) -> List[str]:
    word_set = set(wordDict)
    n = len(s)
    result, path = [], []

    def go(start: int, depth: int) -> None:
        pad = "    " * depth
        if start == n:
            result.append(" ".join(path))
            print(f"{pad}✔ hết chuỗi -> câu {result[-1]!r}")
            return
        found = False
        for end in range(start + 1, n + 1):
            word = s[start:end]
            if word in word_set:
                found = True
                print(f"{pad}start={start}: chọn {word!r} -> đi tới start={end}")
                path.append(word)
                go(end, depth + 1)
                path.pop()
        if not found:
            print(f"{pad}start={start}: hậu tố {s[start:]!r} không bắt đầu bằng từ nào ✘")

    go(0, 0)
    print("Kết quả:", result)
    return result

trace_backtrack("catsanddog", ["cat", "cats", "and", "sand", "dog"])
print("-" * 50)
trace_backtrack("catsandog", ["cats", "dog", "sand", "and", "cat"])

start=0: chọn 'cat' -> đi tới start=3
    start=3: chọn 'sand' -> đi tới start=7
        start=7: chọn 'dog' -> đi tới start=10
            ✔ hết chuỗi -> câu 'cat sand dog'
start=0: chọn 'cats' -> đi tới start=4
    start=4: chọn 'and' -> đi tới start=7
        start=7: chọn 'dog' -> đi tới start=10
            ✔ hết chuỗi -> câu 'cats and dog'
Kết quả: ['cat sand dog', 'cats and dog']
--------------------------------------------------
start=0: chọn 'cat' -> đi tới start=3
    start=3: chọn 'sand' -> đi tới start=7
        start=7: hậu tố 'og' không bắt đầu bằng từ nào ✘
start=0: chọn 'cats' -> đi tới start=4
    start=4: chọn 'and' -> đi tới start=7
        start=7: hậu tố 'og' không bắt đầu bằng từ nào ✘
Kết quả: []


[]

### Đọc trace `"catsanddog"` bằng bảng

| Bước | `start` | Từ chọn | `path` sau khi chọn | Ghi chú |
|---:|---:|---|---|---|
| 1 | 0 | `"cat"` | `["cat"]` | `"c"`, `"ca"` không phải từ → bỏ qua ngay |
| 2 | 3 | `"sand"` | `["cat","sand"]` | |
| 3 | 7 | `"dog"` | `["cat","sand","dog"]` | `start = 10 = n` → ✔ `"cat sand dog"` |
| 4 | 0 | `"cats"` | `["cats"]` | đã `pop` hết, thử tiền tố dài hơn |
| 5 | 4 | `"and"` | `["cats","and"]` | |
| 6 | 7 | `"dog"` | `["cats","and","dog"]` | ✔ `"cats and dog"` |

Lưu ý: ta **đi tới vị trí 7 hai lần** (qua `"cat sand"` và qua `"cats and"`), và lần nào cũng giải lại `"dog"` từ đầu. Với `"catsandog"` thì tệ hơn: ta đi tới vị trí 7 hai lần **chỉ để thấy nhánh chết hai lần**. Đây là mầm mống của vấn đề lớn ở mục kế tiếp.

### Độ phức tạp Cách 1 — và cái bẫy "nhánh chết"

Gọi `n = len(s)`. Trường hợp xấu nhất, số lời gọi `go` có thể tới cỡ `2^n` (mỗi khe hở cắt hoặc không). Mỗi lời gọi lặp `n` giá trị `end` và cắt chuỗi `O(n)`.

- **Time:** `O(2^n · n^2)` trường hợp xấu nhất.
- **Space:** `O(n)` cho `path` + độ sâu đệ quy (chưa tính đầu ra).

Nếu đầu ra thật sự có `2^(n-1)` câu thì tốn thời gian là "đáng" (xem mục 9). Nhưng cái bẫy là: Cách 1 có thể tốn **hàm mũ** thời gian **mà đầu ra rỗng**!

Ví dụ: `s = "aaaa...ab"` (k chữ `a` rồi một chữ `b`), `wordDict = ["a","aa","aaa"]`. Không có từ nào chứa `b` → đáp án là `[]`. Nhưng Cách 1 vẫn thử mọi cách tách đống chữ `a`, và lần nào cũng chết khi chạm `b`. Ta đếm số lời gọi `go`:

In [11]:
def count_calls_backtrack(s: str, wordDict: List[str]) -> int:
    word_set = set(wordDict)
    n = len(s)
    calls = 0

    def go(start: int) -> None:
        nonlocal calls          # nonlocal: sửa biến calls của hàm bên ngoài
        calls += 1
        if start == n:
            return
        for end in range(start + 1, n + 1):
            if s[start:end] in word_set:
                go(end)

    go(0)
    return calls

trap_words = ["a", "aa", "aaa"]
print(" k | số lời gọi go (đáp án luôn là [])")
for k in [6, 10, 14, 18]:
    print(f"{k:>2} | {count_calls_backtrack('a' * k + 'b', trap_words):>8}")

 k | số lời gọi go (đáp án luôn là [])
 6 |       52
10 |      600
14 |     6872
18 |    78652


## 7. Cách 2 — Backtracking + cắt tỉa (pruning) bằng `can_break`

### Cách 1 vướng ở đâu?

Nó bước vào những vị trí mà **hậu tố phía sau không thể tách được** — và cứ thế đào sâu vô ích. Trong ví dụ bẫy, mỗi lời gọi đều đi vào nhánh chết.

### Mấu chốt cải tiến

Tính trước bảng `can = build_can_break(s, word_set)` (Word Break I cho hậu tố, mục 3). Khi xét từ `s[start:end]`, chỉ đi tiếp nếu **cả hai** điều kiện đúng:

```python
if can[end] and s[start:end] in word_set:
    go(end)
```

Hệ quả rất mạnh: **mọi lời gọi `go` đều nằm trên đường dẫn tới ít nhất một câu hợp lệ**. Không còn nhánh chết nào. Ta gọi thuật toán như vậy là **output-sensitive** (nhạy với đầu ra): thời gian tỉ lệ với **kích thước đầu ra**, không phải với số cách cắt.

Thêm một kiểm tra ở đầu: nếu `can[0]` là `False` → trả `[]` ngay, không cần đệ quy.

In [12]:
def word_break_pruned(s: str, wordDict: List[str]) -> List[str]:
    word_set = set(wordDict)
    n = len(s)
    can = build_can_break(s, word_set)
    result, path = [], []
    if not can[0]:
        return result

    def go(start: int) -> None:
        if start == n:
            result.append(" ".join(path))
            return
        for end in range(start + 1, n + 1):
            if can[end] and s[start:end] in word_set:   # chỉ bước vào nhánh "còn sống"
                path.append(s[start:end])
                go(end)
                path.pop()

    go(0)
    return result

for s, words, expected in examples:
    got = word_break_pruned(s, words)
    print(f"{s:<18} -> {got}")
    assert same_answer(got, expected)

catsanddog         -> ['cat sand dog', 'cats and dog']
pineapplepenapple  -> ['pine apple pen apple', 'pine applepen apple', 'pineapple pen apple']
catsandog          -> []


### Đếm lại trên ví dụ bẫy

Ta viết bản đếm lời gọi của Cách 2. Với `"a"*k + "b"`, `can[0] = False` nên ta trả `[]` ngay — **0 lời gọi đệ quy** dù `k` lớn cỡ nào. Còn chi phí tính `can` chỉ là `O(n^2)` cặp `(i, end)`.

Ta cũng thử một ví dụ "nửa sống nửa chết": `s = "a"*k + "b"` nhưng thêm từ `"ab"` vào từ điển. Giờ có đáp án, và Cách 2 chỉ đi vào những nhánh kết thúc bằng `"ab"`.

In [13]:
def count_calls_pruned(s: str, wordDict: List[str]) -> int:
    word_set = set(wordDict)
    n = len(s)
    can = build_can_break(s, word_set)
    calls = 0
    if not can[0]:
        return calls

    def go(start: int) -> None:
        nonlocal calls
        calls += 1
        if start == n:
            return
        for end in range(start + 1, n + 1):
            if can[end] and s[start:end] in word_set:
                go(end)

    go(0)
    return calls

print(" k | Cách 1 (bẫy) | Cách 2 (bẫy) || Cách 1 (+'ab') | Cách 2 (+'ab') | số câu")
for k in [6, 10, 14]:
    s = "a" * k + "b"
    w2 = trap_words + ["ab"]
    print(f"{k:>2} | {count_calls_backtrack(s, trap_words):>12} | {count_calls_pruned(s, trap_words):>12} ||"
          f" {count_calls_backtrack(s, w2):>14} | {count_calls_pruned(s, w2):>14} | {len(word_break_pruned(s, w2)):>6}")

 k | Cách 1 (bẫy) | Cách 2 (bẫy) || Cách 1 (+'ab') | Cách 2 (+'ab') | số câu
 6 |           52 |            0 ||             65 |             41 |     13
10 |          600 |            0 ||            749 |            475 |    149
14 |         6872 |            0 ||           8577 |           5441 |   1705


### Cách 2 còn vướng gì? — bài con bị giải lặp lại

Cắt tỉa đã loại hết nhánh chết, nhưng nhánh **sống** vẫn bị đi lại nhiều lần. Trong Ví dụ 2 `"pineapplepenapple"`: hậu tố `"penapple"` (bắt đầu ở vị trí 9) được tới từ cả `"pine apple"` lẫn `"pineapple"`, và mỗi lần tới, ta lại liệt kê lại toàn bộ các cách tách `"penapple"`.

Hàm dưới đếm **mỗi vị trí `start` được ghé bao nhiêu lần** trong Cách 2.

In [14]:
from collections import Counter   # Counter: dict đếm số lần xuất hiện

def visits_per_start(s: str, wordDict: List[str]) -> Counter:
    word_set = set(wordDict)
    n = len(s)
    can = build_can_break(s, word_set)
    visits = Counter()

    def go(start: int) -> None:
        visits[start] += 1
        if start == n:
            return
        for end in range(start + 1, n + 1):
            if can[end] and s[start:end] in word_set:
                go(end)

    if can[0]:
        go(0)
    return visits

v = visits_per_start("pineapplepenapple", ["apple", "pen", "applepen", "pine", "pineapple"])
print("pineapplepenapple:", dict(sorted(v.items())))

v = visits_per_start("a" * 10, ["a", "aa", "aaa"])
print("'a'*10           :", dict(sorted(v.items())))

pineapplepenapple: {0: 1, 4: 1, 9: 2, 12: 3, 17: 3}
'a'*10           : {0: 1, 1: 1, 2: 2, 3: 4, 4: 7, 5: 13, 6: 24, 7: 44, 8: 81, 9: 149, 10: 274}


Với `"a"*10`, vị trí 10 (cuối chuỗi) được ghé **274 lần** — tức có 274 câu — và các vị trí ở giữa cũng bị ghé hàng chục lần. Mỗi lần ghé lại, ta lại dựng lại cùng một "cây con" lựa chọn.

**Ý tưởng tiếp theo:** đáp án cho hậu tố `s[start:]` **không phụ thuộc** ta đã đi tới `start` bằng đường nào. Vậy hãy tính **một lần**, **ghi nhớ**, rồi dùng lại. Đó là **memoization**.

## 8. Kiến thức nền: memoization (ghi nhớ kết quả)

**Memoization** = trước khi tính `f(x)`, kiểm tra xem đã từng tính chưa; nếu rồi thì trả kết quả đã lưu; nếu chưa thì tính, **lưu vào `memo`**, rồi trả về.

### Ví dụ mini: dãy Fibonacci

`fib(n) = fib(n-1) + fib(n-2)`, với `fib(0) = 0`, `fib(1) = 1`. Đệ quy thuần gọi `fib(n-2)` hai lần (một lần trực tiếp, một lần bên trong `fib(n-1)`), cứ thế nhân lên → số lời gọi tăng theo hàm mũ. Với memo, mỗi `n` chỉ tính đúng 1 lần.

### `functools.lru_cache`: memo tự động

Python có sẵn **decorator** (một "lớp bọc" đặt bằng `@` ngay trên `def`) tên `lru_cache` trong module `functools`. Viết `@lru_cache(maxsize=None)` trên một hàm thì Python tự lưu kết quả theo **tham số**: lần gọi sau với cùng tham số sẽ trả ngay kết quả cũ.

Điều kiện: tham số phải **hashable** (băm được) như `int`, `str`, `tuple` — không dùng được `list`/`dict` làm tham số. Trong bài này tham số là `start: int` nên rất hợp.

In [15]:
calls_plain = 0
def fib_plain(n: int) -> int:
    global calls_plain
    calls_plain += 1
    if n < 2:
        return n
    return fib_plain(n - 1) + fib_plain(n - 2)

calls_memo = 0
def fib_memo(n: int, memo: dict) -> int:
    global calls_memo
    calls_memo += 1
    if n in memo:                  # 1. đã tính rồi -> dùng lại
        return memo[n]
    if n < 2:
        result = n
    else:
        result = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    memo[n] = result               # 2. lưu lại trước khi trả về
    return result

print("fib(25) =", fib_plain(25), "| số lời gọi không memo:", calls_plain)
print("fib(25) =", fib_memo(25, {}), "| số lời gọi có memo  :", calls_memo)

# Cùng ý tưởng, nhưng để Python tự memo bằng lru_cache (xem giải thích ở cell markdown phía trên)
from functools import lru_cache

@lru_cache(maxsize=None)
def fib_cached(n: int) -> int:
    if n < 2:
        return n
    return fib_cached(n - 1) + fib_cached(n - 2)

print("fib(80) =", fib_cached(80))
print(fib_cached.cache_info())   # hits = số lần dùng lại, misses = số lần tính mới

fib(25) = 75025 | số lời gọi không memo: 242785
fib(25) = 75025 | số lời gọi có memo  : 49
fib(80) = 23416728348467685
CacheInfo(hits=78, misses=81, maxsize=None, currsize=81)


## 9. Cách 3 — Đệ quy + memo: `sentences(start)` trả về **list câu** (cách chính)

### Đổi góc nhìn: hàm **trả về** kết quả thay vì "ghi vào `path`"

Ở Cách 1–2, hàm `go` không trả gì, chỉ ghi câu vào `result` khi tới cuối. Muốn memo được, ta cần hàm **trả về đáp án của bài con**:

> `sentences(start)` = **list tất cả các câu** tạo từ hậu tố `s[start:]`.

**Công thức (recurrence):** với mỗi `end` mà `word = s[start:end]` là từ, mỗi câu `rest` trong `sentences(end)` cho ta một câu mới:

```text
câu mới = word                 nếu rest == ""   (word là từ cuối cùng)
        = word + " " + rest    nếu rest != ""
```

**Base case:** `sentences(n) = [""]` — một list chứa **đúng một câu rỗng**.

### Vì sao base case là `[""]` chứ không phải `[]`?

Đây là chỗ người mới hay sai nhất, nên ta nói kỹ:

- `[]` nghĩa là "**không có cách nào**" tách hậu tố.
- `[""]` nghĩa là "**có đúng một cách**: dùng 0 từ".

Hậu tố rỗng **tách được** (bằng 0 từ), giống `dp[0] = True` ở Word Break I. Nếu ta trả `[]`, thì vòng `for rest in sentences(n)` không chạy lần nào → từ cuối như `"dog"` không bao giờ tạo được câu → mọi đáp án đều rỗng. (Mục 16 có code minh họa lỗi này.)

### Memo giải quyết luôn cả 2 vấn đề của Cách 1

- **Bài con lặp lại:** `sentences(7)` của `"catsanddog"` chỉ tính 1 lần, lần sau lấy từ `memo`.
- **Nhánh chết:** nếu `sentences(i) = []`, kết quả `[]` cũng được lưu — lần sau chạm vào `i` thì trả `[]` ngay, không đào lại. Vì vậy Cách 3 **không cần** bảng `can_break` riêng.

In [16]:
def word_break_memo(s: str, wordDict: List[str]) -> List[str]:
    word_set = set(wordDict)
    n = len(s)
    memo = {}                                   # memo[start] = list câu của s[start:]

    def sentences(start: int) -> List[str]:
        if start in memo:
            return memo[start]
        if start == n:
            return [""]                         # đúng 1 cách: 0 từ
        result = []
        for end in range(start + 1, n + 1):
            word = s[start:end]
            if word in word_set:
                for rest in sentences(end):
                    result.append(word if rest == "" else word + " " + rest)
        memo[start] = result
        return result

    return sentences(0)

for s, words, expected in examples:
    got = word_break_memo(s, words)
    print(f"{s:<18} -> {got}")
    assert same_answer(got, expected)

# Dùng lại từ nhiều lần + đoạn chết ở giữa
print(word_break_memo("aaaa", ["a", "aa"]))
print(word_break_memo("abcd", ["a", "abc", "b", "cd"]))

catsanddog         -> ['cat sand dog', 'cats and dog']
pineapplepenapple  -> ['pine apple pen apple', 'pine applepen apple', 'pineapple pen apple']
catsandog          -> []
['a a a a', 'a a aa', 'a aa a', 'aa a a', 'aa aa']
['a b cd']


### Trace Cách 3: memo được điền theo thứ tự nào?

`trace_memo` in khi **vào** `sentences(start)`, khi **dùng lại memo** (♻), và khi **trả về** (lưu memo). Thụt lề theo độ sâu. Hàm trả về chính `memo` để ta nhìn bảng cuối cùng.

Chú ý: memo được điền **từ phải sang trái** — vị trí xa nhất (gần cuối chuỗi) trả về trước, rồi kết quả "nổi" ngược lên vị trí 0. Đây gọi là thứ tự **post-order** (con xong rồi mới tới cha).

In [17]:
def trace_memo(s: str, wordDict: List[str], verbose: bool = True) -> dict:
    word_set = set(wordDict)
    n = len(s)
    memo = {n: [""]}                            # điền sẵn base case để in cho gọn

    def sentences(start: int, depth: int) -> List[str]:
        pad = "   " * depth
        if start in memo:
            if verbose and start != n:
                print(f"{pad}♻ dùng lại memo[{start}] = {memo[start]}")
            return memo[start]
        if verbose:
            print(f"{pad}→ vào sentences({start}), hậu tố {s[start:]!r}")
        result = []
        for end in range(start + 1, n + 1):
            word = s[start:end]
            if word in word_set:
                for rest in sentences(end, depth + 1):
                    result.append(word if rest == "" else word + " " + rest)
        memo[start] = result
        if verbose:
            print(f"{pad}← lưu memo[{start}] = {result}")
        return result

    sentences(0, 0)
    return memo

memo = trace_memo("catsanddog", ["cat", "cats", "and", "sand", "dog"])
print("\nBảng memo cuối:")
for k in sorted(memo):
    print(f"  memo[{k:>2}] = {memo[k]}")

→ vào sentences(0), hậu tố 'catsanddog'
   → vào sentences(3), hậu tố 'sanddog'
      → vào sentences(7), hậu tố 'dog'
      ← lưu memo[7] = ['dog']
   ← lưu memo[3] = ['sand dog']
   → vào sentences(4), hậu tố 'anddog'
      ♻ dùng lại memo[7] = ['dog']
   ← lưu memo[4] = ['and dog']
← lưu memo[0] = ['cat sand dog', 'cats and dog']

Bảng memo cuối:
  memo[ 0] = ['cat sand dog', 'cats and dog']
  memo[ 3] = ['sand dog']
  memo[ 4] = ['and dog']
  memo[ 7] = ['dog']
  memo[10] = ['']


### Đọc trace `"catsanddog"` bằng bảng

Chỉ số: `c0 a1 t2 s3 a4 n5 d6 d7 o8 g9`, `n = 10`.

| Thứ tự lưu | `start` | Hậu tố `s[start:]` | Từ đầu hợp lệ | `memo[start]` |
|---:|---:|---|---|---|
| (sẵn) | 10 | `""` | — | `[""]` |
| 1 | 7 | `"dog"` | `"dog"` → `memo[10]` | `["dog"]` |
| 2 | 3 | `"sanddog"` | `"sand"` → `memo[7]` | `["sand dog"]` |
| 3 | 4 | `"anddog"` | `"and"` → `memo[7]` ♻ | `["and dog"]` |
| 4 | 0 | `"catsanddog"` | `"cat"` → `memo[3]`, `"cats"` → `memo[4]` | `["cat sand dog", "cats and dog"]` |

Dòng 3 là điểm mấu chốt: khi tới vị trí 7 lần thứ hai, ta **không giải lại** `"dog"` mà lấy ngay `memo[7]`.

Các vị trí như 1, 2, 5, 6, 8, 9 **không bao giờ được gọi**, vì không có từ nào kết thúc đúng ở đó xuất phát từ một vị trí đang xét. Memo chỉ chứa những vị trí thật sự "đi tới được".

### Trace thêm: dùng lại memo **rỗng** (nhánh chết) trong `"catsandog"`

Ở `"catsandog"`, vị trí 7 (hậu tố `"og"`) là nhánh chết: `memo[7] = []`. Khi đi tới 7 lần thứ hai (qua `"cats and"`), ta dùng lại `[]` ngay — không đào lại.

In [18]:
memo = trace_memo("catsandog", ["cats", "dog", "sand", "and", "cat"])
print("\nKết quả memo[0] =", memo[0])

→ vào sentences(0), hậu tố 'catsandog'
   → vào sentences(3), hậu tố 'sandog'
      → vào sentences(7), hậu tố 'og'
      ← lưu memo[7] = []
   ← lưu memo[3] = []
   → vào sentences(4), hậu tố 'andog'
      ♻ dùng lại memo[7] = []
   ← lưu memo[4] = []
← lưu memo[0] = []

Kết quả memo[0] = []


### Memo có thực sự bớt việc không? Đếm số lần **thân hàm** chạy

Ta so sánh trên hai input:

- **Bẫy** `"a"*18 + "b"` với `["a","aa","aaa"]`: Cách 1 chạy hàng chục nghìn lần; Cách 3 mỗi vị trí chỉ tính 1 lần.
- **Nhiều đáp án** `"a"*12` với `["a","aa","aaa"]`: Cách 2 ghé lại mỗi vị trí rất nhiều lần; Cách 3 tính mỗi vị trí 1 lần (nhưng vẫn phải **dựng** đủ các câu — phần này không tránh được).

In [19]:
def count_body_runs_memo(s: str, wordDict: List[str]) -> int:
    word_set = set(wordDict)
    n = len(s)
    memo = {}
    runs = 0

    def sentences(start: int) -> List[str]:
        nonlocal runs
        if start in memo:
            return memo[start]
        runs += 1                               # chỉ đếm khi thật sự tính
        if start == n:
            memo[start] = [""]
            return memo[start]
        result = []
        for end in range(start + 1, n + 1):
            word = s[start:end]
            if word in word_set:
                for rest in sentences(end):
                    result.append(word if rest == "" else word + " " + rest)
        memo[start] = result
        return result

    sentences(0)
    return runs

trap = "a" * 18 + "b"
print("Bẫy  : Cách 1 =", count_calls_backtrack(trap, trap_words),
      "| Cách 3 =", count_body_runs_memo(trap, trap_words))
many = "a" * 12
print("'a'*12: Cách 2 =", count_calls_pruned(many, trap_words),
      "| Cách 3 =", count_body_runs_memo(many, trap_words),
      "| số câu =", len(word_break_memo(many, trap_words)))

Bẫy  : Cách 1 = 78652 | Cách 3 = 19
'a'*12: Cách 2 = 2031 | Cách 3 = 13 | số câu = 927


### Vì sao đầu ra có thể lớn theo hàm mũ? (và vì sao không có thuật toán đa thức tổng quát)

Xét `s = "aaa...a"` (`n` chữ `a`) và từ điển chứa `"a"`, `"aa"`, `"aaa"`, ..., `"aaaaaaaaaa"` (10 từ — đề giới hạn độ dài từ ≤ 10). Khi `n ≤ 10`, **mọi** cách cắt đều hợp lệ → có đúng `2^(n-1)` câu (mục 4). Khi `n > 10`, chỉ những cách cắt có mảnh dài hơn 10 bị loại, nên số câu **hơi ít hơn** `2^(n-1)` (ví dụ `n = 12`: 2045 thay vì 2048) — nhưng vẫn **tăng theo hàm mũ**. Mỗi câu dài khoảng `n` ký tự, nên **chỉ riêng việc in đáp án** đã tốn cỡ `n · 2^(n-1)` thao tác.

Vì vậy:

- **Không thể** có thuật toán chạy trong thời gian đa thức (`O(n^k)`) cho **mọi** input: đầu ra đã lớn hơn mọi đa thức.
- Điều tốt nhất có thể làm là thuật toán **output-sensitive**: thời gian ≈ `O(đa thức(n) + kích thước đầu ra)`. Cách 2 và Cách 3 đều đạt tinh thần này; Cách 1 thì không (nó có thể tốn hàm mũ cả khi đầu ra rỗng).
- Đây chính là lý do đề phải **hứa** "độ dài đáp án không vượt quá `10^5`" và giới hạn `s.length <= 20`.

Ngược lại, **đếm** số câu (không liệt kê) thì làm được trong thời gian đa thức: `count[i] = tổng count[end]` với mọi từ `s[i:end]` — chỉ cộng số, không dựng chuỗi. "Đếm thì nhanh, liệt kê thì không" — một bài học quan trọng của dạng bài này.

In [20]:
def count_sentences(s: str, wordDict: List[str]) -> int:
    # Đếm số câu bằng DP từ phải sang trái — O(n^2) phép cắt chuỗi, không dựng câu
    word_set = set(wordDict)
    n = len(s)
    count = [0] * (n + 1)
    count[n] = 1                       # 1 cách cho hậu tố rỗng (giống [""])
    for i in range(n - 1, -1, -1):
        for end in range(i + 1, n + 1):
            if s[i:end] in word_set:
                count[i] += count[end]
    return count[0]

all_a = ["a" * k for k in range(1, 11)]          # "a", "aa", ..., 10 chữ a (đúng giới hạn độ dài từ)
print(" n | số câu (liệt kê) | đếm bằng DP | 2^(n-1)")
for n in [1, 2, 3, 4, 8, 12]:
    listed = len(word_break_memo("a" * n, all_a))
    print(f"{n:>2} | {listed:>16} | {count_sentences('a' * n, all_a):>11} | {2 ** (n - 1):>7}")

print("Với n = 20, chỉ ĐẾM đã thấy:", count_sentences("a" * 20, all_a), "câu (gần 2^19 = 524288)")

 n | số câu (liệt kê) | đếm bằng DP | 2^(n-1)
 1 |                1 |           1 |       1
 2 |                2 |           2 |       2
 3 |                4 |           4 |       4
 4 |                8 |           8 |       8
 8 |              128 |         128 |     128
12 |             2045 |        2045 |    2048
Với n = 20, chỉ ĐẾM đã thấy: 521472 câu (gần 2^19 = 524288)


### Độ phức tạp Cách 3

Gọi `n = len(s)`, `S` = **tổng số ký tự** của mọi câu được lưu trong `memo` (tức tổng kích thước các đáp án của mọi hậu tố đi tới được).

- **Số trạng thái:** `n + 1` vị trí `start`, mỗi vị trí tính **đúng một lần**.
- **Kiểm tra từ:** mỗi vị trí thử `n` giá trị `end`, mỗi lần cắt + băm chuỗi `O(n)` → `O(n^3)` (với `n ≤ 20` là rất nhỏ). Nếu chỉ thử `end ≤ start + max_len` (độ dài từ dài nhất, ≤ 10) thì còn `O(n · max_len^2)`.
- **Dựng câu:** mỗi câu trong `memo[start]` được tạo bằng một phép nối chuỗi tốn `O(độ dài câu)` → tổng `O(S)`.

**Time:** `O(n^3 + S)`. Trường hợp xấu nhất (`"aaa...a"`), `S` cỡ `O(n · 2^n)`.

**Space:** `O(S)` cho memo + `O(n)` độ sâu đệ quy. Trường hợp xấu nhất cũng `O(n · 2^n)`.

So với Cách 2, memo tốn **thêm bộ nhớ** (lưu đáp án của mọi hậu tố), nhưng đổi lại mỗi bài con chỉ giải một lần, code ngắn và tự động xử lý nhánh chết. Với ràng buộc của đề (đáp án ≤ `10^5` ký tự), đây là lựa chọn gọn và an toàn nhất.

## 10. Cách 4 — DP bottom-up: `dp[i]` = list câu của **tiền tố** `s[:i]`

Đây là "anh em" của Word Break I (mục 3), nhưng thay `True/False` bằng **list câu**:

- `dp[0] = [""]` (tiền tố rỗng có 1 cách: 0 từ).
- Với `i` từ 1 tới `n`: với mỗi `j < i` mà `s[j:i]` là từ, mỗi câu `prev` trong `dp[j]` cho câu mới `prev + " " + s[j:i]` (hoặc chỉ `s[j:i]` nếu `prev == ""`).
- Đáp án là `dp[n]`.

Khác Cách 3 ở chỗ: Cách 3 đi **từ trên xuống** (top-down, gọi đệ quy từ `start = 0`), còn Cách 4 đi **từ dưới lên** (bottom-up, điền bảng từ `i = 0` tới `n` bằng vòng lặp, không đệ quy).

**Bẫy cú pháp:** tạo bảng bằng `[[] for _ in range(n + 1)]`, **không** dùng `[[]] * (n + 1)` (mọi ô sẽ là **cùng một list** — xem mục 16).

In [21]:
def word_break_dp(s: str, wordDict: List[str]) -> List[str]:
    word_set = set(wordDict)
    n = len(s)
    dp = [[] for _ in range(n + 1)]     # mỗi ô là một list RIÊNG
    dp[0] = [""]
    for i in range(1, n + 1):
        for j in range(i):
            word = s[j:i]
            if dp[j] and word in word_set:
                for prev in dp[j]:
                    dp[i].append(word if prev == "" else prev + " " + word)
    return dp[n]

for s, words, expected in examples:
    got = word_break_dp(s, words)
    print(f"{s:<18} -> {got}")
    assert same_answer(got, expected)

catsanddog         -> ['cat sand dog', 'cats and dog']
pineapplepenapple  -> ['pine applepen apple', 'pineapple pen apple', 'pine apple pen apple']
catsandog          -> []


### Trace Cách 4: in các ô `dp[i]` khác rỗng

Với `"catsanddog"` và `"catsandog"`. Hãy để ý ô `dp[7]` của `"catsandog"`: ta đã **dựng sẵn** 2 câu `"cat sand"`, `"cats and"` — nhưng cuối cùng không dùng được vì phần `"og"` không tách được.

In [22]:
def trace_dp(s: str, wordDict: List[str]) -> List[str]:
    word_set = set(wordDict)
    n = len(s)
    dp = [[] for _ in range(n + 1)]
    dp[0] = [""]
    for i in range(1, n + 1):
        for j in range(i):
            word = s[j:i]
            if dp[j] and word in word_set:
                for prev in dp[j]:
                    dp[i].append(word if prev == "" else prev + " " + word)
        if dp[i]:
            print(f"dp[{i:>2}] ({s[:i]!r:<13}) = {dp[i]}")
    print("Đáp án dp[n] =", dp[n])
    return dp[n]

trace_dp("catsanddog", ["cat", "cats", "and", "sand", "dog"])
print("-" * 50)
trace_dp("catsandog", ["cats", "dog", "sand", "and", "cat"])

dp[ 3] ('cat'        ) = ['cat']
dp[ 4] ('cats'       ) = ['cats']
dp[ 7] ('catsand'    ) = ['cat sand', 'cats and']
dp[10] ('catsanddog' ) = ['cat sand dog', 'cats and dog']
Đáp án dp[n] = ['cat sand dog', 'cats and dog']
--------------------------------------------------
dp[ 3] ('cat'        ) = ['cat']
dp[ 4] ('cats'       ) = ['cats']
dp[ 7] ('catsand'    ) = ['cat sand', 'cats and']
Đáp án dp[n] = []


[]

### Cách 4 vướng ở đâu? — dựng câu cho tiền tố "cụt"

Cách 4 dựng câu cho **mọi tiền tố tách được**, kể cả khi phần **sau** nó không bao giờ tách được. Với bẫy `"a"*k + "b"`, mọi tiền tố `"aaa..."` đều tách được theo rất nhiều cách → Cách 4 dựng ra **hàng chục nghìn** câu dở dang, để rồi `dp[n] = []`.

Cách 3 không bị vậy vì nó xây câu **từ hậu tố**: một hậu tố chỉ có câu khi nó **tách được tới tận cuối chuỗi**.

Cách sửa cho Cách 4 (nếu thích bottom-up): kết hợp với `can_break` — chỉ thêm câu vào `dp[i]` khi `can[i]` (hậu tố từ `i`) cũng `True`. Ta đo số câu dở dang được tạo ra:

In [23]:
def total_partial_sentences_dp(s: str, wordDict: List[str], use_can: bool) -> int:
    word_set = set(wordDict)
    n = len(s)
    can = build_can_break(s, word_set) if use_can else [True] * (n + 1)
    dp = [[] for _ in range(n + 1)]
    dp[0] = [""] if can[0] else []
    for i in range(1, n + 1):
        if not can[i]:
            continue                      # tiền tố s[:i] không thể nối tiếp tới cuối -> bỏ
        for j in range(i):
            word = s[j:i]
            if dp[j] and word in word_set:
                for prev in dp[j]:
                    dp[i].append(word if prev == "" else prev + " " + word)
    return sum(len(cell) for cell in dp)

trap = "a" * 18 + "b"
print("Số câu (kể cả dở dang) Cách 4 tạo ra trên bẫy:", total_partial_sentences_dp(trap, trap_words, use_can=False))
print("Cách 4 + can_break                          :", total_partial_sentences_dp(trap, trap_words, use_can=True))

Số câu (kể cả dở dang) Cách 4 tạo ra trên bẫy: 78652
Cách 4 + can_break                          : 0


### Độ phức tạp Cách 4

- **Time:** `O(n^3 + P)`, với `P` = tổng số ký tự của **mọi câu dựng cho mọi tiền tố** — có thể lớn theo hàm mũ **ngay cả khi đáp án rỗng** (như bẫy ở trên). Nếu thêm `can_break`, `P` chỉ còn các tiền tố "còn đường đi tới cuối" và trở lại output-sensitive.
- **Space:** `O(P)`.

Bài học: với dạng "liệt kê tất cả", **top-down + memo** thường tự nhiên hơn bottom-up vì nó chỉ đi vào những trạng thái thật sự cần.

## 11. Cách 5 (tùy chọn) — Trie (cây tiền tố) để duyệt tiền tố

### Vòng lặp `end` của Cách 3 còn phí ở đâu?

Tại mỗi `start`, ta thử **mọi** `end` từ `start+1` tới `n`, mỗi lần cắt ra `s[start:end]` rồi băm để tra `set`. Ví dụ với `start = 0` của `"catsanddog"`, ta tra cả `"catsa"`, `"catsan"`, ..., `"catsanddog"` — dù **không có từ nào** bắt đầu bằng `"catsa"`. Biết được điều đó thì nên **dừng sớm**.

### Trie là gì?

**Trie** (đọc là "try", còn gọi **cây tiền tố — prefix tree**) là cây mà mỗi cạnh là **một ký tự**; đi từ gốc xuống theo các ký tự `c → a → t` ta đọc được tiền tố `"cat"`. Nút nào đánh dấu "kết thúc từ" nghĩa là đường đi tới nó là một từ trong từ điển.

Trong Python, cài Trie gọn nhất bằng **dict lồng nhau**: mỗi nút là một `dict` ánh xạ `ký tự → nút con`, và khóa đặc biệt `"$"` đánh dấu kết thúc từ.

```text
(gốc)
 ├─ c ─ a ─ t ($) ─ s ($)        "cat", "cats"
 ├─ a ─ n ─ d ($)                "and"
 ├─ s ─ a ─ n ─ d ($)            "sand"
 └─ d ─ o ─ g ($)                "dog"
```

Để tìm **mọi từ bắt đầu tại `start`**, ta đi trong Trie theo `s[start], s[start+1], ...`; gặp `"$"` là có một từ; **không còn nút con khớp ký tự tiếp theo thì dừng luôn**.

In [24]:
END = "$"

def build_trie(words: List[str]) -> dict:
    root = {}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})   # có nút con thì đi, chưa có thì tạo
        node[END] = True                     # đánh dấu: tới đây là một từ
    return root

def word_ends_from(s: str, start: int, trie: dict) -> List[int]:
    # Trả về mọi end sao cho s[start:end] là từ — dừng ngay khi Trie hết đường
    ends = []
    node = trie
    for i in range(start, len(s)):
        ch = s[i]
        if ch not in node:
            break
        node = node[ch]
        if END in node:
            ends.append(i + 1)
    return ends

trie = build_trie(["cat", "cats", "and", "sand", "dog"])
print("Các nhánh con của gốc:", list(trie.keys()))
print("Nút sau 'c','a','t'  :", trie["c"]["a"]["t"])
s = "catsanddog"
for start in [0, 3, 4, 7]:
    ends = word_ends_from(s, start, trie)
    print(f"start={start}: từ bắt đầu ở đây ->", [s[start:e] for e in ends])

Các nhánh con của gốc: ['c', 'a', 's', 'd']
Nút sau 'c','a','t'  : {'$': True, 's': {'$': True}}
start=0: từ bắt đầu ở đây -> ['cat', 'cats']
start=3: từ bắt đầu ở đây -> ['sand']
start=4: từ bắt đầu ở đây -> ['and']
start=7: từ bắt đầu ở đây -> ['dog']


### Ghép Trie vào Cách 3

Chỉ thay vòng `for end in range(start + 1, n + 1)` + `if word in word_set` bằng `for end in word_ends_from(s, start, trie)`. Phần memo giữ nguyên.

**Độ phức tạp:** xây Trie tốn `O(tổng độ dài các từ)` (≤ 1000 · 10). Mỗi `start` chỉ đi tối đa `max_len` ký tự trong Trie → phần kiểm tra từ còn `O(n · max_len)` thay vì `O(n^3)`. Phần dựng câu `O(S)` không đổi.

Với `n ≤ 20` của bài này, Trie không làm khác biệt đáng kể; nhưng nó là kỹ thuật **rất đáng biết** cho các bài chuỗi dài / nhiều từ (ví dụ Word Search II, Concatenated Words).

In [25]:
def word_break_trie(s: str, wordDict: List[str]) -> List[str]:
    trie = build_trie(wordDict)
    n = len(s)
    memo = {n: [""]}

    def sentences(start: int) -> List[str]:
        if start in memo:
            return memo[start]
        result = []
        for end in word_ends_from(s, start, trie):
            word = s[start:end]
            for rest in sentences(end):
                result.append(word if rest == "" else word + " " + rest)
        memo[start] = result
        return result

    return sentences(0)

for s, words, expected in examples:
    got = word_break_trie(s, words)
    print(f"{s:<18} -> {got}")
    assert same_answer(got, expected)

catsanddog         -> ['cat sand dog', 'cats and dog']
pineapplepenapple  -> ['pine apple pen apple', 'pine applepen apple', 'pineapple pen apple']
catsandog          -> []


## 12. Code nộp LeetCode — Cách 3 (đệ quy + memo)

Chuyển `word_break_memo` sang đúng format LeetCode:

- Giữ nguyên signature `def wordBreak(self, s: str, wordDict: List[str]) -> List[str]`.
- Trên LeetCode, `List` đã được import sẵn; khi chạy trong notebook ta cần `from typing import List` (đã có ở đầu).
- Dùng `@lru_cache` cho hàm con `sentences(start)`. Vì hàm con được định nghĩa **bên trong** `wordBreak`, cache được tạo mới cho mỗi lần gọi → không lẫn kết quả giữa các test.
- Thêm tối ưu nhỏ: chỉ thử `end ≤ start + max_len` (độ dài từ dài nhất), vì đoạn dài hơn chắc chắn không phải từ.
- Hàm con trả về **tuple** thay vì list để không ai vô tình sửa kết quả đã cache (mẹo an toàn với `lru_cache`).

In [26]:
from functools import lru_cache
from typing import List


class Solution:
    def wordBreak(self, s: str, wordDict: List[str]) -> List[str]:
        word_set = set(wordDict)
        max_len = max(len(w) for w in wordDict)
        n = len(s)

        @lru_cache(maxsize=None)
        def sentences(start: int) -> tuple:
            if start == n:
                return ("",)
            result = []
            for end in range(start + 1, min(n, start + max_len) + 1):
                word = s[start:end]
                if word in word_set:
                    for rest in sentences(end):
                        result.append(word if rest == "" else word + " " + rest)
            return tuple(result)

        return list(sentences(0))

### Bộ test bằng `assert`

Ta kiểm tra: 3 ví dụ của đề, và các edge case:

1. `s` dài 1 ký tự, có / không có trong từ điển.
2. Cả `s` là đúng một từ (đáp án 1 câu, không có dấu cách).
3. Dùng lại một từ nhiều lần (`"aaaa"` với `["a","aa"]` → 5 câu).
4. Từ điển có từ dài hơn `s`.
5. Không có từ nào khớp ký tự cuối (đáp án rỗng).
6. Trường hợp gần lớn nhất: `"a"*20` với 10 từ `"a"`..`"aaaaaaaaaa"` — chỉ kiểm tra **số câu** bằng `count_sentences`.

In [27]:
def check(name, actual, expected):
    assert same_answer(actual, expected), (name, sorted(actual), sorted(expected))
    print(f"PASS  {name}")

sol = Solution()
for idx, (s, words, expected) in enumerate(examples, 1):
    check(f"ví dụ {idx}", sol.wordBreak(s, words), expected)

check("1 ký tự có trong từ điển", sol.wordBreak("a", ["a"]), ["a"])
check("1 ký tự không có", sol.wordBreak("a", ["b"]), [])
check("cả s là một từ", sol.wordBreak("leetcode", ["leetcode"]), ["leetcode"])
check("dùng lại từ", sol.wordBreak("aaaa", ["a", "aa"]),
      ["a a a a", "aa a a", "a aa a", "a a aa", "aa aa"])
check("từ dài hơn s", sol.wordBreak("ab", ["abc", "a", "b"]), ["a b"])
check("ký tự cuối không khớp", sol.wordBreak("aaab", ["a", "aa"]), [])
check("đoạn chết ở giữa", sol.wordBreak("abcd", ["a", "abc", "b", "cd"]), ["a b cd"])

big = sol.wordBreak("a" * 20, all_a)
assert len(big) == count_sentences("a" * 20, all_a) and len(set(big)) == len(big)
print(f"PASS  'a'*20: {len(big)} câu, không trùng lặp")

PASS  ví dụ 1
PASS  ví dụ 2
PASS  ví dụ 3
PASS  1 ký tự có trong từ điển
PASS  1 ký tự không có
PASS  cả s là một từ
PASS  dùng lại từ
PASS  từ dài hơn s
PASS  ký tự cuối không khớp
PASS  đoạn chết ở giữa


PASS  'a'*20: 521472 câu, không trùng lặp


## 13. Test đối chiếu chéo trên input ngẫu nhiên

Ta sinh nhiều input nhỏ với `random.seed` cố định (để ai chạy lại cũng ra cùng kết quả):

- `s` dài 1..12, chỉ gồm `'a'` và `'b'` (bảng chữ nhỏ → dễ có nhiều cách tách và nhiều nhánh chết).
- `wordDict` là vài từ ngẫu nhiên dài 1..3 (không trùng nhau, đúng ràng buộc đề).

Mọi cách (Brute force, Cách 1, 2, 3, 4, 5, `Solution`) phải cho **cùng một tập câu**, và số câu phải bằng `count_sentences`.

In [28]:
import random
from itertools import product

random.seed(140)
alphabet = "ab"
pool = ["".join(p) for L in range(1, 4) for p in product(alphabet, repeat=L)]   # 14 từ có thể

methods = {
    "backtrack": word_break_backtrack,
    "pruned": word_break_pruned,
    "memo": word_break_memo,
    "dp": word_break_dp,
    "trie": word_break_trie,
    "Solution": lambda s, w: Solution().wordBreak(s, w),
}

total_sentences = 0
empty_cases = 0
for _ in range(400):
    s = "".join(random.choice(alphabet) for _ in range(random.randint(1, 12)))
    words = random.sample(pool, random.randint(2, 8))
    expected = word_break_brute(s, words)
    for name, fn in methods.items():
        got = fn(s, words)
        assert same_answer(got, expected), (name, s, words, got, expected)
    assert len(expected) == count_sentences(s, words)
    total_sentences += len(expected)
    empty_cases += (len(expected) == 0)

print("400 test ngẫu nhiên: 6 cách + count_sentences khớp brute force")
print("Tổng số câu đã so:", total_sentences, "| số test có đáp án rỗng:", empty_cases)

400 test ngẫu nhiên: 6 cách + count_sentences khớp brute force
Tổng số câu đã so: 1452 | số test có đáp án rỗng: 218


## 14. Vì sao thuật toán đúng? (Cách 3)

**Khẳng định (invariant):** với mọi `start` từ `n` về `0`, `sentences(start)` trả về **đúng** tập tất cả các câu hợp lệ của hậu tố `s[start:]` — **không thiếu**, **không thừa**, **không trùng**.

Chứng minh theo kiểu **quy nạp** (đi từ hậu tố ngắn đến hậu tố dài):

- **Base:** `start = n`, hậu tố rỗng chỉ có một cách tách là "0 từ" → `[""]`. Đúng.
- **Bước:** giả sử khẳng định đúng cho mọi vị trí `> start`. Xét một câu hợp lệ bất kỳ của `s[start:]`:
  - Nó có **từ đầu tiên** `w = s[start:end]` (với `end > start`, `w` thuộc từ điển), và phần còn lại là một câu hợp lệ `rest` của `s[end:]`.
  - Theo giả thiết quy nạp, `rest` có trong `sentences(end)`, và vòng lặp sẽ thử đúng `end` đó → câu này **được tạo ra**. ⇒ **Không thiếu.**
  - Ngược lại, mỗi câu thuật toán tạo ra đều là "một từ hợp lệ + một câu hợp lệ của phần sau" → **hợp lệ**. ⇒ **Không thừa.**
  - Hai câu khác nhau sinh ra từ hoặc **từ đầu khác nhau** (khác `end`), hoặc **cùng từ đầu nhưng `rest` khác nhau** → chuỗi kết quả khác nhau. ⇒ **Không trùng.**
- **Memo không làm sai:** kết quả của `sentences(start)` chỉ phụ thuộc `start` (và `s`, từ điển — cố định trong một lần gọi), **không** phụ thuộc đường đi tới `start`, nên dùng lại là an toàn.

**Cách 2 đúng vì:** cắt tỉa chỉ bỏ những `end` mà `can[end] = False`, tức hậu tố `s[end:]` **không có câu nào** → không bỏ sót câu nào.

## 15. So sánh các cách

Ký hiệu: `n = len(s)`, `S` = tổng độ dài đáp án của các hậu tố (≈ kích thước đầu ra), `P` = tổng độ dài các câu dở dang cho mọi tiền tố.

| Cách | Ý tưởng | Time | Space (ngoài đầu ra) | Nhánh chết? | Nhận xét |
|---|---|---|---|---|---|
| 0. Brute force | Sinh `2^(n-1)` cách cắt rồi lọc | `O(2^n · n)` luôn luôn | `O(2^n · n)` | Đi hết | Chỉ dùng làm trọng tài |
| 1. Backtracking | Chỉ đi tiếp khi tiền tố là từ | `O(2^n · n^2)` xấu nhất | `O(n)` | Bị đi lại hàm mũ lần | Đơn giản, dính bẫy `"aa...ab"` |
| 2. Backtracking + `can_break` | Chỉ vào nhánh còn sống | `O(n^2 + n · số câu · n)` | `O(n)` | Không vào | Output-sensitive, ít bộ nhớ |
| **3. Đệ quy + memo** | `sentences(start)` = list câu hậu tố | `O(n^3 + S)` | `O(S)` | Tính 1 lần, lưu `[]` | **Cách chính**, gọn, chuẩn phỏng vấn |
| 4. DP bottom-up | `dp[i]` = list câu tiền tố | `O(n^3 + P)` | `O(P)` | Dựng câu dở dang | Cần thêm `can_break` để khỏi phí |
| 5. Memo + Trie | Duyệt tiền tố bằng Trie | `O(Σ|w| + n · max_len + S)` | `O(Σ|w| + S)` | Như Cách 3 | Có lợi khi từ điển / chuỗi lớn |

Trường hợp xấu nhất của **mọi** cách đều là hàm mũ vì **đầu ra** có thể lớn theo hàm mũ (mục 9).

## 16. Các lỗi người mới hay gặp

### Lỗi 1: Base case trả `[]` thay vì `[""]`

Sai:

```python
if start == n:
    return []          # "không có cách nào" — SAI ý nghĩa!
```

Vòng `for rest in sentences(n)` không chạy lần nào → từ cuối cùng không bao giờ tạo được câu → **mọi** đáp án rỗng.

### Lỗi 2: Luôn nối `word + " " + rest` → thừa dấu cách ở cuối

Với base case `[""]`, nếu nối máy móc `word + " " + rest` thì từ cuối thành `"dog" + " " + "" = "dog "` — câu bị **thừa dấu cách cuối**, LeetCode chấm sai. Phải xử lý riêng `rest == ""` (hoặc dùng `" ".join(...)` trên list từ như Cách 1).

Cell dưới chạy cả hai hàm sai cạnh hàm đúng để thấy khác biệt (dùng `repr` để nhìn rõ dấu cách cuối).

In [29]:
def wrong_base_case(s: str, wordDict: List[str]) -> List[str]:
    word_set = set(wordDict)
    n = len(s)
    memo = {}

    def sentences(start: int) -> List[str]:
        if start in memo:
            return memo[start]
        if start == n:
            return []                           # LỖI: phải là [""]
        result = []
        for end in range(start + 1, n + 1):
            word = s[start:end]
            if word in word_set:
                for rest in sentences(end):
                    result.append(word if rest == "" else word + " " + rest)
        memo[start] = result
        return result

    return sentences(0)



def wrong_trailing_space(s: str, wordDict: List[str]) -> List[str]:
    word_set = set(wordDict)
    n = len(s)

    def sentences(start: int) -> List[str]:
        if start == n:
            return [""]
        result = []
        for end in range(start + 1, n + 1):
            word = s[start:end]
            if word in word_set:
                for rest in sentences(end):
                    result.append(word + " " + rest)        # LỖI: không xét rest == ""
        return result

    return sentences(0)

w1 = ["cat", "cats", "and", "sand", "dog"]
print("Lỗi 1 (base [])      :", wrong_base_case("catsanddog", w1))
print("Lỗi 2 (thừa dấu cách):", [repr(x) for x in wrong_trailing_space("catsanddog", w1)])
print("Đúng                 :", [repr(x) for x in word_break_memo("catsanddog", w1)])

Lỗi 1 (base [])      : []
Lỗi 2 (thừa dấu cách): ["'cat sand dog '", "'cats and dog '"]
Đúng                 : ["'cat sand dog'", "'cats and dog'"]


### Lỗi 3: Dùng tham số mặc định `memo={}` — memo bị **dùng chung giữa các lần gọi**

Trong Python, giá trị mặc định của tham số (như `memo={}`) chỉ được tạo **một lần** khi định nghĩa hàm, rồi **dùng chung** cho mọi lần gọi sau. Memo theo khóa `start` của chuỗi thứ nhất sẽ bị dùng nhầm cho chuỗi thứ hai! Trên LeetCode, nhiều test chạy liên tiếp nên lỗi này cho kết quả sai rất khó hiểu.

Sửa: tạo `memo = {}` **bên trong** hàm ngoài (như `word_break_memo`), hoặc dùng `@lru_cache` cho hàm con định nghĩa bên trong.

In [30]:
def wrong_shared_memo(s: str, wordDict: List[str], start: int = 0, memo={}) -> List[str]:
    if start in memo:                          # LỖI: memo còn sót từ lần gọi trước
        return memo[start]
    if start == len(s):
        return [""]
    result = []
    for end in range(start + 1, len(s) + 1):
        word = s[start:end]
        if word in wordDict:
            for rest in wrong_shared_memo(s, wordDict, end, memo):
                result.append(word if rest == "" else word + " " + rest)
    memo[start] = result
    return result

print("Lần 1:", wrong_shared_memo("catsanddog", ["cat", "cats", "and", "sand", "dog"]))
print("Lần 2:", wrong_shared_memo("ab", ["a", "b"]), " <- sai! đúng phải là", word_break_memo("ab", ["a", "b"]))

Lần 1: ['cat sand dog', 'cats and dog']
Lần 2: ['cat sand dog', 'cats and dog']  <- sai! đúng phải là ['a b']


### Lỗi 4: Tạo bảng DP bằng `[[]] * (n + 1)`

`[[]] * 3` tạo ra một list chứa **3 tham chiếu tới cùng một list rỗng**. `append` vào `dp[1]` thì `dp[0]`, `dp[2]` cũng "đổi theo". Cách 4 sẽ ra kết quả lộn xộn.

Sửa: `dp = [[] for _ in range(n + 1)]` — mỗi lần lặp tạo một list **mới**.

In [31]:
bad = [[]] * 3
bad[1].append("cat")
print("bad :", bad, "| cùng một list?", bad[0] is bad[2])

good = [[] for _ in range(3)]
good[1].append("cat")
print("good:", good, "| cùng một list?", good[0] is good[2])

bad : [['cat'], ['cat'], ['cat']] | cùng một list? True
good: [[], ['cat'], []] | cùng một list? False


### Lỗi 5: Quên rằng thứ tự đáp án là tùy ý

Tự test bằng `assert result == expected` sẽ báo sai dù thuật toán đúng, vì thứ tự câu khác. Luôn so sánh bằng `sorted(...)` (hàm `same_answer` ở mục 1).

### Lỗi 6: Dùng `list` để tra từ

`word in wordDict` với `wordDict` là list 1000 phần tử phải dò tuần tự. Chạy vẫn đúng, nhưng chậm hơn nhiều so với `set`. Luôn `word_set = set(wordDict)` trước.

### Lỗi 7: Sửa list đã lưu trong memo

Nếu ở đâu đó ta `memo[end].append(...)` hoặc sửa list trả về từ hàm memo, các vị trí khác đang dùng chung list đó sẽ bị hỏng. Luôn tạo `result = []` **mới** cho mỗi `start`; với `lru_cache`, trả về `tuple` (bất biến — immutable) cho an toàn.

## 17. Mẹo nhận dạng pattern

- Đề nói "**trả về tất cả** cách / mọi tổ hợp / mọi câu" → nghĩ tới **backtracking**; nếu bài con (hậu tố, đoạn con) lặp lại → **backtracking + memo**.
- Đề chỉ hỏi "**có tồn tại không**" / "**bao nhiêu cách**" / "**ít nhất bao nhiêu**" → thường là **DP đa thức** (Word Break I, đếm câu, câu ít từ nhất).
- "Tách chuỗi thành các mảnh thuộc một tập" → trạng thái thường chỉ là **một chỉ số** (`start` hoặc `i`).
- Thấy ràng buộc nhỏ bất thường (`n ≤ 20`) kèm "đáp án ≤ `10^5`" → tín hiệu đầu ra có thể **lớn theo hàm mũ**, đề đang giới hạn để liệt kê được.
- Muốn tránh nhánh chết khi liệt kê → tính trước bảng "**còn đường tới đích không**" (`can_break`), rồi chỉ đi vào nhánh sống.
- Từ điển lớn / cần duyệt tiền tố nhanh → **Trie**.

## 18. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| 139. Word Break | Chỉ hỏi `True/False` — DP `dp[i]` (mục 3) |
| 472. Concatenated Words | Mỗi từ có phải ghép từ ≥ 2 từ ngắn hơn trong list? — Word Break I cho từng từ (Bài 6) |
| 131. Palindrome Partitioning | Cùng khung "chọn mảnh đầu + đệ quy phần sau", điều kiện là palindrome thay vì thuộc từ điển |
| 93. Restore IP Addresses | Liệt kê mọi cách cắt chuỗi thành 4 mảnh hợp lệ |
| 2707. Extra Characters in a String | DP tối ưu số ký tự thừa khi tách chuỗi theo từ điển |
| 212. Word Search II | Dùng Trie để duyệt tiền tố nhiều từ cùng lúc |

## 19. Bài tập vận dụng

### Bài 1 — Đoán output (không chạy code)

Với `word_break_memo`, mỗi lời gọi sau trả về bao nhiêu câu? Viết ra các câu.

```python
word_break_memo("aaa", ["a", "aa"])
word_break_memo("catsandog", ["cats", "dog", "sand", "and", "cat"])
word_break_memo("abab", ["ab", "a", "b", "ba"])
```

### Bài 2 — Tính tay bảng memo

Với Ví dụ 2: `s = "pineapplepenapple"`, `wordDict = ["apple","pen","applepen","pine","pineapple"]`.

1. Viết chỉ số bắt đầu của mỗi từ trong `s` (ví dụ `"pine"` là `s[0:4]`).
2. Điền bảng `memo[start]` cho các vị trí được gọi tới, theo **thứ tự được lưu** (từ phải sang trái): `start = 17, 12, 9, 4, 0`.
3. Vị trí nào được dùng lại từ memo (♻)?

### Bài 3 — Điền TODO: Word Break I (bottom-up)

Hoàn thiện hàm trả về `True` nếu `s` tách được.

In [32]:
def exercise_word_break_i(s: str, wordDict: List[str]) -> bool:
    word_set = set(wordDict)
    n = len(s)
    dp = [False] * (n + 1)
    # TODO 1: đặt điểm tựa cho tiền tố rỗng
    # TODO 2: với mỗi i từ 1..n, tìm j < i sao cho dp[j] đúng và s[j:i] là từ
    # TODO 3: trả về dp[n]
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_word_break_i("leetcode", ["leet", "code"]) == True
# assert exercise_word_break_i("applepenapple", ["apple", "pen"]) == True
# assert exercise_word_break_i("catsandog", ["cats", "dog", "sand", "and", "cat"]) == False

### Bài 4 — Tự viết lại Cách 3 (đệ quy + memo)

Không nhìn lên trên, tự viết `exercise_word_break_ii`. Nhớ 3 điểm: `set` cho từ điển, base case `[""]`, và xử lý `rest == ""` khi nối câu.

In [33]:
def exercise_word_break_ii(s: str, wordDict: List[str]) -> List[str]:
    # TODO: memo theo start; sentences(start) trả về list câu của s[start:]
    raise NotImplementedError

# Gợi ý test:
# assert same_answer(exercise_word_break_ii("catsanddog", ["cat", "cats", "and", "sand", "dog"]),
#                    ["cats and dog", "cat sand dog"])
# assert exercise_word_break_ii("catsandog", ["cats", "dog", "sand", "and", "cat"]) == []
# assert len(exercise_word_break_ii("aaaa", ["a", "aa"])) == 5

### Bài 5 — Biến thể: câu có **ít từ nhất**

Viết `exercise_min_words(s, wordDict)` trả về **số từ ít nhất** cần để tách `s`, hoặc `-1` nếu không tách được. Yêu cầu: chạy **đa thức** (không liệt kê các câu!).

Ví dụ: `"catsanddog"` → `3`; `"pineapplepenapple"` → `3` (`"pineapple pen apple"`); `"catsandog"` → `-1`.

In [34]:
def exercise_min_words(s: str, wordDict: List[str]) -> int:
    # TODO: best[i] = số từ ít nhất để tách s[i:]  (hoặc vô cực nếu không tách được)
    pass

# Gợi ý test:
# assert exercise_min_words("catsanddog", ["cat", "cats", "and", "sand", "dog"]) == 3
# assert exercise_min_words("pineapplepenapple", ["apple", "pen", "applepen", "pine", "pineapple"]) == 3
# assert exercise_min_words("catsandog", ["cats", "dog", "sand", "and", "cat"]) == -1
# assert exercise_min_words("a" * 20, ["a" * k for k in range(1, 11)]) == 2

### Bài 6 — Mini challenge: Concatenated Words (LeetCode 472)

Cho list từ `words` (không trùng nhau). Trả về mọi từ trong `words` mà **ghép được từ ít nhất 2 từ ngắn hơn** cũng nằm trong `words` (mỗi từ ngắn có thể dùng lại).

```text
Input:  ["cat","cats","catsdogcats","dog","dogcatsdog","hippopotamuses","rat","ratcatdogcat"]
Output: ["catsdogcats","dogcatsdog","ratcatdogcat"]
```

Yêu cầu: tái sử dụng ý tưởng Word Break I. Cẩn thận: một từ không được "tự ghép bằng chính nó" (`"cat"` không tính là ghép từ `"cat"`).

In [35]:
def exercise_concatenated_words(words: List[str]) -> List[str]:
    # TODO: với mỗi từ w, kiểm tra w có tách được thành >= 2 từ của những từ KHÁC w không
    pass

# Gợi ý test:
# words = ["cat", "cats", "catsdogcats", "dog", "dogcatsdog", "hippopotamuses", "rat", "ratcatdogcat"]
# assert sorted(exercise_concatenated_words(words)) == ["catsdogcats", "dogcatsdog", "ratcatdogcat"]
# assert exercise_concatenated_words(["cat", "dog", "catdog"]) == ["catdog"]
# assert exercise_concatenated_words(["a"]) == []

## 20. Gợi ý theo mức

**Bài 1**
- Mức 1: Liệt kê cây lựa chọn từ vị trí 0, mỗi bước chọn một từ là tiền tố của phần còn lại.
- Mức 2: `"aaa"` với `{"a","aa"}`: số câu tuân theo kiểu Fibonacci (`count[i] = count[i+1] + count[i+2]`).
- Mức 3: `"abab"`: thử từ đầu `"a"` rồi `"ab"`; sau `"a"` còn `"bab"` → `"b"`/`"ba"`...

**Bài 2**
- Mức 1: Tìm mọi vị trí một từ bắt đầu và kết thúc: `"pine"` 0→4, `"pineapple"` 0→9, ...
- Mức 2: `memo[17] = [""]`; vị trí 12 chỉ có `"apple"` → `memo[12] = ["apple"]`.
- Mức 3: Vị trí 9 được tới từ `"apple"` (4→9) và từ `"pineapple"` (0→9).

**Bài 3**
- Mức 1: Điểm tựa là `dp[0]`.
- Mức 2: Hai vòng lặp `for i in range(1, n + 1)`, `for j in range(i)`.
- Mức 3: `if dp[j] and s[j:i] in word_set: dp[i] = True; break`.

**Bài 4**
- Mức 1: Hàm con `sentences(start)`, kiểm tra `start in memo` đầu tiên.
- Mức 2: Base case `start == n` → `[""]`.
- Mức 3: `result.append(word if rest == "" else word + " " + rest)`, cuối cùng `memo[start] = result`.

**Bài 5**
- Mức 1: Giống `count_sentences` nhưng thay "cộng" bằng "lấy min" và "+1 từ".
- Mức 2: `best[n] = 0`; `best[i] = min(best[i], best[end] + 1)` khi `s[i:end]` là từ.
- Mức 3: Dùng `float("inf")` làm vô cực; cuối cùng nếu `best[0]` là vô cực thì trả `-1`.

**Bài 6**
- Mức 1: Từ `w` là "ghép" nếu Word Break I của `w` đúng khi từ điển là **các từ khác** `w`.
- Mức 2: Bỏ `w` khỏi `set` trước khi kiểm tra rồi thêm lại; hoặc chỉ dùng các từ **ngắn hơn** `w`.
- Mức 3: Vì mọi mảnh đều ngắn hơn `w` (nếu ≥ 2 mảnh), chỉ cần cấm `j == 0 and i == len(w)` trong DP; bỏ qua từ rỗng.

## 21. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

- `"aaa"`, `["a","aa"]` → **3 câu**: `"a a a"`, `"a aa"`, `"aa a"`.
- `"catsandog"` → **0 câu**: `[]`.
- `"abab"`, `["ab","a","b","ba"]` → **5 câu**: `"a b a b"`, `"a b ab"`, `"a ba b"`, `"ab a b"`, `"ab ab"`. Người mới hay quên `"a ba b"` (từ `"ba"` nằm "vắt" qua ranh giới hai cặp `ab`) hoặc `"ab ab"`.

### Đáp án Bài 2

Vị trí các từ: `"pine"` 0→4, `"pineapple"` 0→9, `"apple"` 4→9, `"applepen"` 4→12, `"pen"` 9→12, `"apple"` 12→17.

| Thứ tự lưu | `start` | `memo[start]` |
|---:|---:|---|
| (sẵn) | 17 | `[""]` |
| 1 | 12 | `["apple"]` |
| 2 | 9 | `["pen apple"]` |
| 3 | 4 | `["apple pen apple", "applepen apple"]` (dùng lại `memo[12]` ♻ khi thử `"applepen"`) |
| 4 | 0 | `["pine apple pen apple", "pine applepen apple", "pineapple pen apple"]` (dùng lại `memo[9]` ♻ khi thử `"pineapple"`) |

Hai lần dùng lại memo: `memo[12]` (từ vị trí 4 qua `"applepen"`) và `memo[9]` (từ vị trí 0 qua `"pineapple"`).

In [36]:
# Kiểm chứng Bài 1
for s, words in [("aaa", ["a", "aa"]),
                 ("catsandog", ["cats", "dog", "sand", "and", "cat"]),
                 ("abab", ["ab", "a", "b", "ba"])]:
    got = word_break_memo(s, words)
    print(f"{s!r}: {len(got)} câu -> {got}")

# Kiểm chứng Bài 2 (in trace để thấy ♻)
print()
memo = trace_memo("pineapplepenapple", ["apple", "pen", "applepen", "pine", "pineapple"])
assert memo[12] == ["apple"] and memo[9] == ["pen apple"]
assert same_answer(memo[4], ["apple pen apple", "applepen apple"])
assert same_answer(memo[0], ["pine apple pen apple", "pine applepen apple", "pineapple pen apple"])
print("Bài 2 PASS")

'aaa': 3 câu -> ['a a a', 'a aa', 'aa a']
'catsandog': 0 câu -> []
'abab': 5 câu -> ['a b a b', 'a b ab', 'a ba b', 'ab a b', 'ab ab']

→ vào sentences(0), hậu tố 'pineapplepenapple'
   → vào sentences(4), hậu tố 'applepenapple'
      → vào sentences(9), hậu tố 'penapple'
         → vào sentences(12), hậu tố 'apple'
         ← lưu memo[12] = ['apple']
      ← lưu memo[9] = ['pen apple']
      ♻ dùng lại memo[12] = ['apple']
   ← lưu memo[4] = ['apple pen apple', 'applepen apple']
   ♻ dùng lại memo[9] = ['pen apple']
← lưu memo[0] = ['pine apple pen apple', 'pine applepen apple', 'pineapple pen apple']
Bài 2 PASS


In [37]:
# ===== Đáp án Bài 3 và Bài 4 =====
def answer_word_break_i(s: str, wordDict: List[str]) -> bool:
    word_set = set(wordDict)
    n = len(s)
    dp = [False] * (n + 1)
    dp[0] = True
    for i in range(1, n + 1):
        for j in range(i):
            if dp[j] and s[j:i] in word_set:
                dp[i] = True
                break
    return dp[n]


def answer_word_break_ii(s: str, wordDict: List[str]) -> List[str]:
    word_set = set(wordDict)
    n = len(s)
    memo = {}

    def sentences(start: int) -> List[str]:
        if start in memo:
            return memo[start]
        if start == n:
            return [""]
        result = []
        for end in range(start + 1, n + 1):
            word = s[start:end]
            if word in word_set:
                for rest in sentences(end):
                    result.append(word if rest == "" else word + " " + rest)
        memo[start] = result
        return result

    return sentences(0)


assert answer_word_break_i("leetcode", ["leet", "code"]) == True
assert answer_word_break_i("applepenapple", ["apple", "pen"]) == True
assert answer_word_break_i("catsandog", ["cats", "dog", "sand", "and", "cat"]) == False
assert same_answer(answer_word_break_ii("catsanddog", ["cat", "cats", "and", "sand", "dog"]),
                   ["cats and dog", "cat sand dog"])
assert answer_word_break_ii("catsandog", ["cats", "dog", "sand", "and", "cat"]) == []
assert len(answer_word_break_ii("aaaa", ["a", "aa"])) == 5
print("Bài 3, Bài 4 PASS")

Bài 3, Bài 4 PASS


In [38]:
# ===== Đáp án Bài 5 và Bài 6 =====
def answer_min_words(s: str, wordDict: List[str]) -> int:
    word_set = set(wordDict)
    n = len(s)
    INF = float("inf")
    best = [INF] * (n + 1)
    best[n] = 0                                   # hậu tố rỗng: 0 từ
    for i in range(n - 1, -1, -1):
        for end in range(i + 1, n + 1):
            if s[i:end] in word_set and best[end] + 1 < best[i]:
                best[i] = best[end] + 1
    return -1 if best[0] == INF else best[0]


def answer_concatenated_words(words: List[str]) -> List[str]:
    word_set = set(words)
    result = []
    for w in words:
        n = len(w)
        if n == 0:
            continue
        dp = [False] * (n + 1)
        dp[0] = True
        for i in range(1, n + 1):
            for j in range(i):
                if j == 0 and i == n:             # cấm dùng chính w làm một mảnh
                    continue
                if dp[j] and w[j:i] in word_set:
                    dp[i] = True
                    break
        if dp[n]:
            result.append(w)
    return result


assert answer_min_words("catsanddog", ["cat", "cats", "and", "sand", "dog"]) == 3
assert answer_min_words("pineapplepenapple", ["apple", "pen", "applepen", "pine", "pineapple"]) == 3
assert answer_min_words("catsandog", ["cats", "dog", "sand", "and", "cat"]) == -1
assert answer_min_words("a" * 20, ["a" * k for k in range(1, 11)]) == 2

words = ["cat", "cats", "catsdogcats", "dog", "dogcatsdog", "hippopotamuses", "rat", "ratcatdogcat"]
assert sorted(answer_concatenated_words(words)) == ["catsdogcats", "dogcatsdog", "ratcatdogcat"]
assert answer_concatenated_words(["cat", "dog", "catdog"]) == ["catdog"]
assert answer_concatenated_words(["a"]) == []

# Đối chiếu Bài 5 với cách liệt kê (chỉ trên input nhỏ)
random.seed(5)
for _ in range(200):
    s = "".join(random.choice("ab") for _ in range(random.randint(1, 10)))
    ws = random.sample(pool, random.randint(1, 5))
    listed = word_break_memo(s, ws)
    expect = min((len(x.split()) for x in listed), default=-1)
    assert answer_min_words(s, ws) == expect
print("Bài 5, Bài 6 PASS (kèm 200 test đối chiếu cho Bài 5)")

Bài 5, Bài 6 PASS (kèm 200 test đối chiếu cho Bài 5)


## 22. Template ngắn nên nhớ

```python
class Solution:
    def wordBreak(self, s: str, wordDict: List[str]) -> List[str]:
        word_set = set(wordDict)
        n = len(s)
        memo = {}

        def sentences(start):                 # list câu của hậu tố s[start:]
            if start in memo:
                return memo[start]
            if start == n:
                return [""]                   # 1 cách: 0 từ  (KHÔNG phải [])
            result = []
            for end in range(start + 1, n + 1):
                word = s[start:end]
                if word in word_set:
                    for rest in sentences(end):
                        result.append(word if rest == "" else word + " " + rest)
            memo[start] = result
            return result

        return sentences(0)
```

Câu thần chú: **"Chọn từ đầu tiên — ghép với mọi câu của phần còn lại — nhớ kết quả theo vị trí."**

Và nhớ: **đếm / kiểm tra tồn tại thì đa thức; liệt kê tất cả thì có thể hàm mũ.**

## 23. Checklist tự kiểm tra trước khi submit

- Mình đã đổi `wordDict` sang `set` chưa?
- Base case hậu tố rỗng có trả `[""]` (một câu rỗng) chứ **không phải** `[]` chưa?
- Khi nối câu có xử lý `rest == ""` để **không thừa dấu cách cuối** chưa?
- Memo có được tạo **mới cho mỗi lần gọi** `wordBreak` (không dùng `memo={}` làm tham số mặc định) chưa?
- Có lỡ **sửa** list đã nằm trong memo không? (mỗi `start` một `result = []` mới; với `lru_cache` trả `tuple`)
- Nếu dùng DP bottom-up: tạo bảng bằng `[[] for _ in range(n + 1)]` chưa? Có cắt tỉa tiền tố "cụt" bằng `can_break` không?
- Đã test: 3 ví dụ đề, `s` 1 ký tự, cả `s` là một từ, từ dùng lại nhiều lần (`"aaaa"`), đáp án rỗng, bẫy `"aa...ab"`?
- Khi tự test đã so sánh **bỏ qua thứ tự** (`sorted`) chưa?